In [1]:
!pip install control


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.3/578.3 kB 9.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import control as ctrl

# Buck converter parameters
Vin = 10
Vout = 5
L = 1e-3
C = 10e-6
R = 0.1

# Transfer function
num = [Vout*L/R, Vout*(1 - R*C/L), 0]
den = [L*C, R*C, 1]
G = ctrl.TransferFunction(num, den)

# ACO parameters
num_ants = 10
num_iterations = 50
evaporation_rate = 0.5
alpha = 1
beta = 1

# Initialize pheromones
pheromones = np.ones((2, num_iterations))

# Store best values
Kp_list = []
Ki_list = []
performance_list = []

for iteration in range(num_iterations):
    for ant in range(num_ants):
        # Random PI parameters
        Kp = np.random.rand()
        Ki = np.random.rand()

        # PI Controller: Kp*(s + Ki)/s
        PI = Kp * ctrl.TransferFunction([1, Ki], [1, 0])

        # Closed-loop system
        sys = ctrl.feedback(G, PI)

        # Step response info
        t, y = ctrl.step_response(sys)

        # Calculate settling time and overshoot manually
        y_final = y[-1]
        overshoot = (np.max(y) - y_final) / y_final * 100 if y_final != 0 else 0

        # Settling time (2% band)
        within_band = np.where(np.abs(y - y_final) <= 0.02 * abs(y_final))[0]
        settling_time = t[within_band[0]] if len(within_band) > 0 else t[-1]

        # Performance
        performance = alpha * settling_time + beta * overshoot

        # Store values
        Kp_list.append(Kp)
        Ki_list.append(Ki)
        performance_list.append(performance)

        # Update pheromones
        pheromones[:, iteration] = (1 - evaporation_rate) * pheromones[:, iteration] + performance

# Find best parameters
best_index = np.argmin(performance_list)
best_Kp = Kp_list[best_index]
best_Ki = Ki_list[best_index]

print("Best PI Controller Parameters:")
print("Kp =", best_Kp)
print("Ki =", best_Ki)

Best PI Controller Parameters:
Kp = 0.016380973863800108
Ki = 0.1569362899767056
